# J3-PM — PyTorch : MLP vs CNN sur MNIST

**Objectif :** comparer un MLP et un CNN de taille similaire (~100K paramètres) pour montrer l'apport de la convolution à budget constant.

| Modèle | Architecture | Params |
|--------|-------------|--------|
| MLP | 784 → 128 → 64 → 10 | ~109 K |
| CNN | Conv(16)+Pool → Conv(32)+Pool → FC(64) → 10 | ~106 K |


In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')


## 1. Chargement de MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

print(f'Train : {len(train_ds):,} exemples  |  Test : {len(test_ds):,} exemples')
print(f'Shape : {train_ds[0][0].shape}  dtype : {train_ds[0][0].dtype}')


In [ ]:
# Grille 2×10 : train (haut) / test (bas)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    for row, ds in enumerate([train_ds, test_ds]):
        img, label = ds[i]
        axes[row, i].imshow(img.squeeze(), cmap='gray')
        axes[row, i].set_title(label)
        axes[row, i].axis('off')
axes[0, 0].set_ylabel('train', fontsize=9)
axes[1, 0].set_ylabel('test',  fontsize=9)
plt.suptitle('Exemples MNIST', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


## 2. Modèle MLP

Architecture **784 → 128 → 64 → 10** (~109 K paramètres).

In [ ]:
class MLP(nn.Module):
    """MLP : 784 -> 128 -> 64 -> 10  (~109 K paramètres)"""

    def __init__(self) -> None:
        """Entrée : x [B, 1, 28, 28]  |  Sortie : logits [B, 10]."""
        super().__init__()
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x [B, 1, 28, 28]  →  logits [B, 10]."""
        x = x.view(x.size(0), -1)     # Flatten [B,1,28,28] -> [B,784]
        raise NotImplementedError


In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


mlp = MLP().to(DEVICE)
print(f'MLP : {count_params(mlp):,} paramètres')


## 3. Boucle d'Entraînement

In [ ]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
) -> float:
    """model + loader (une époque)  →  float : loss cross-entropy moyenne."""
    model.train()
    total = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        raise NotImplementedError
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    """model + loader  →  float : accuracy ∈ [0, 1]."""
    model.eval()
    correct = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        raise NotImplementedError
    return correct / len(loader.dataset)


In [ ]:
N_EPOCHS  = 10
criterion = nn.CrossEntropyLoss()
opt_mlp   = optim.Adam(mlp.parameters(), lr=1e-3)

history: dict[str, list[float]] = {'mlp_loss': [], 'mlp_acc': []}

print(f'{"Epoch":>5}  {"MLP loss":>10}  {"MLP acc":>9}')
print('-' * 33)
for epoch in range(1, N_EPOCHS + 1):
    l_mlp = train_epoch(mlp, train_loader, opt_mlp, criterion)
    a_mlp = evaluate(mlp, test_loader)
    history['mlp_loss'].append(l_mlp)
    history['mlp_acc'].append(a_mlp)
    print(f'{epoch:5d}  {l_mlp:10.4f}  {a_mlp*100:8.2f}%')


In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['mlp_loss'], 'o-', color='steelblue')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Loss (cross-entropy)')
ax1.set_title('Perte MLP — train')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [a * 100 for a in history['mlp_acc']], 'o-', color='steelblue')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Précision MLP — test')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'MLP accuracy finale : {history["mlp_acc"][-1] * 100:.2f}%')


## 4. CNN : Comparaison

Budget similaire (~100K paramètres) pour une comparaison équitable.

In [ ]:
class CNN(nn.Module):
    """CNN : Conv(16)+Pool -> Conv(32)+Pool -> FC(64) -> 10  (~106 K paramètres)"""

    def __init__(self) -> None:
        """Entrée : x [B, 1, 28, 28]  |  Sortie : logits [B, 10]."""
        super().__init__()
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x [B, 1, 28, 28]  →  logits [B, 10]."""
        raise NotImplementedError


In [ ]:
cnn = CNN().to(DEVICE)
n_mlp, n_cnn = count_params(mlp), count_params(cnn)
print(f'MLP : {n_mlp:,} paramètres')
print(f'CNN : {n_cnn:,} paramètres')
print(f'Ratio CNN/MLP : {n_cnn / n_mlp:.3f}')


In [ ]:
opt_cnn = optim.Adam(cnn.parameters(), lr=1e-3)
history['cnn_loss'] = []
history['cnn_acc']  = []

print(f'{"Epoch":>5}  {"CNN loss":>10}  {"CNN acc":>9}')
print('-' * 33)
for epoch in range(1, N_EPOCHS + 1):
    l_cnn = train_epoch(cnn, train_loader, opt_cnn, criterion)
    a_cnn = evaluate(cnn, test_loader)
    history['cnn_loss'].append(l_cnn)
    history['cnn_acc'].append(a_cnn)
    print(f'{epoch:5d}  {l_cnn:10.4f}  {a_cnn*100:8.2f}%')


In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['mlp_loss'], 'o-', label='MLP', color='steelblue')
ax1.plot(epochs, history['cnn_loss'], 's-', label='CNN', color='tomato')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Loss (cross-entropy moyenne)')
ax1.set_title('Perte — train')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [a * 100 for a in history['mlp_acc']], 'o-', label='MLP', color='steelblue')
ax2.plot(epochs, [a * 100 for a in history['cnn_acc']], 's-', label='CNN', color='tomato')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Précision — test')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Résultats finaux (epoch {N_EPOCHS}) :')
print(f'  MLP : {history["mlp_acc"][-1] * 100:.2f}%')
print(f'  CNN : {history["cnn_acc"][-1] * 100:.2f}%')
print(f'  Ecart : +{(history["cnn_acc"][-1] - history["mlp_acc"][-1]) * 100:.2f} pp pour le CNN')


## 5. Erreurs MLP corrigées par le CNN

Exemples **mal classés par le MLP** mais **bien classés par le CNN**.


In [ ]:
@torch.no_grad()
def get_preds(
    model: nn.Module, loader: DataLoader
) -> tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    preds, labels = [], []
    for X, y in loader:
        X = X.to(DEVICE)
        preds.append(model(X).argmax(1).cpu())
        labels.append(y)
    return torch.cat(preds), torch.cat(labels)


preds_mlp, true_labels = get_preds(mlp, test_loader)
preds_cnn, _           = get_preds(cnn, test_loader)

mlp_wrong = preds_mlp != true_labels
cnn_right  = preds_cnn == true_labels
rescued    = torch.where(mlp_wrong & cnn_right)[0]

print(f'Erreurs MLP  : {mlp_wrong.sum().item()}')
print(f'Erreurs CNN  : {(preds_cnn != true_labels).sum().item()}')
print(f'Rescues CNN  : {len(rescued)}')


In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for col, idx in enumerate(rescued[:10]):
    i = idx.item()
    img        = test_ds[i][0].squeeze().numpy()
    true_lbl   = true_labels[i].item()
    wrong_lbl  = preds_mlp[i].item()
    right_lbl  = preds_cnn[i].item()

    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(f'v:{true_lbl} p:{wrong_lbl}', fontsize=8, color='red')
    axes[0, col].axis('off')

    axes[1, col].imshow(img, cmap='gray')
    axes[1, col].set_title(f'v:{true_lbl} p:{right_lbl}', fontsize=8, color='green')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('MLP ✗', fontsize=10, color='red')
axes[1, 0].set_ylabel('CNN ✓', fontsize=10, color='green')
plt.suptitle('Erreurs MLP corrigees par CNN  (v=vrai, p=predit)', fontsize=11)
plt.tight_layout()
plt.show()


## 6. Analyse

**Pourquoi le CNN fait mieux avec le même budget ?**

1. **Biais inductif spatial** : les convolutions exploitent la *localité* — deux pixels voisins sont traités par le même filtre.
2. **Partage des poids** : un filtre 3×3 (9 valeurs) couvre toute l'image. Le MLP doit apprendre des connexions redondantes pour chaque position.
3. **Invariance par translation** : le max-pooling rend le réseau robuste aux petits décalages — ce que ne sait pas faire le MLP.

> Sur CIFAR-10 ou ImageNet, l'écart entre MLP et CNN est bien plus marqué.


## 7. CIFAR-10 : L'Avantage CNN s'Amplifie

CIFAR-10 : images **32×32 RGB**, 10 classes (avion, voiture, oiseau…).
La complexité est bien supérieure à MNIST — les pixels adjacents portent de l'information
spatiale que le MLP ne peut pas exploiter.

| Modèle | Architecture | Params |
|--------|-------------|--------|
| MLP | 3072 → 512 → 256 → 10 | ~1.7 M |
| CNN | Conv(32)×2+Pool → Conv(64)×2+Pool → FC(512) → 10 | ~2.2 M |

In [ ]:
transform_c10 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_c10 = datasets.CIFAR10('./data', train=True,  download=True, transform=transform_c10)
test_c10  = datasets.CIFAR10('./data', train=False, download=True, transform=transform_c10)

train_c10_loader = DataLoader(train_c10, batch_size=256, shuffle=True,  num_workers=2)
test_c10_loader  = DataLoader(test_c10,  batch_size=256, shuffle=False, num_workers=2)

CLASSES_C10 = train_c10.classes
print(f'Train : {len(train_c10):,}  |  Test : {len(test_c10):,}')
print(f'Shape : {train_c10[0][0].shape}  (C×H×W)')
print(f'Classes : {CLASSES_C10}')


In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)
for i in range(10):
    for row, ds in enumerate([train_c10, test_c10]):
        img, label = ds[i]
        img = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row, i].imshow(img)
        axes[row, i].set_title(CLASSES_C10[label], fontsize=7)
        axes[row, i].axis('off')
axes[0, 0].set_ylabel('train', fontsize=9)
axes[1, 0].set_ylabel('test',  fontsize=9)
plt.suptitle('Exemples CIFAR-10', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
class MLP_C10(nn.Module):
    """MLP : 3072 -> 512 -> 256 -> 10  (~1.7 M paramètres)"""

    def __init__(self) -> None:
        """Entrée : x [B, 3, 32, 32]  |  Sortie : logits [B, 10]."""
        super().__init__()
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x [B, 3, 32, 32]  →  logits [B, 10]."""
        x = x.view(x.size(0), -1)
        raise NotImplementedError


class CNN_C10(nn.Module):
    """CNN : Conv(32)×2+Pool -> Conv(64)×2+Pool -> FC(512) -> 10  (~1.2 M paramètres)"""

    def __init__(self) -> None:
        """Entrée : x [B, 3, 32, 32]  |  Sortie : logits [B, 10]."""
        super().__init__()
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x [B, 3, 32, 32]  →  logits [B, 10]."""
        raise NotImplementedError


mlp_c10 = MLP_C10().to(DEVICE)
cnn_c10 = CNN_C10().to(DEVICE)
print(f'MLP_C10 : {count_params(mlp_c10):,} paramètres')
print(f'CNN_C10 : {count_params(cnn_c10):,} paramètres')


In [ ]:
N_C10     = 5
crit_c10  = nn.CrossEntropyLoss()
opt_mlp_c = optim.Adam(mlp_c10.parameters(), lr=1e-3)
opt_cnn_c = optim.Adam(cnn_c10.parameters(), lr=1e-3)

hist_c10: dict[str, list[float]] = {
    'mlp_loss': [], 'cnn_loss': [], 'mlp_acc': [], 'cnn_acc': []
}

print(f'{"Epoch":>5}  {"MLP loss":>10}  {"MLP acc":>9}  {"CNN loss":>10}  {"CNN acc":>9}')
print('-' * 55)
for epoch in range(1, N_C10 + 1):
    l_m = train_epoch(mlp_c10, train_c10_loader, opt_mlp_c, crit_c10)
    l_c = train_epoch(cnn_c10, train_c10_loader, opt_cnn_c, crit_c10)
    a_m = evaluate(mlp_c10, test_c10_loader)
    a_c = evaluate(cnn_c10, test_c10_loader)
    hist_c10['mlp_loss'].append(l_m)
    hist_c10['cnn_loss'].append(l_c)
    hist_c10['mlp_acc'].append(a_m)
    hist_c10['cnn_acc'].append(a_c)
    print(f'{epoch:5d}  {l_m:10.4f}  {a_m*100:8.2f}%  {l_c:10.4f}  {a_c*100:8.2f}%')


In [ ]:
ep = range(1, N_C10 + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ep, hist_c10['mlp_loss'], 'o-', label='MLP',  color='steelblue')
ax1.plot(ep, hist_c10['cnn_loss'], 's-', label='CNN',  color='tomato')
ax1.set_xlabel('Époque')
ax1.set_ylabel('Loss (cross-entropy moyenne)')
ax1.set_title('Perte — CIFAR-10 train')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ep, [a * 100 for a in hist_c10['mlp_acc']], 'o-', label='MLP',  color='steelblue')
ax2.plot(ep, [a * 100 for a in hist_c10['cnn_acc']], 's-', label='CNN',  color='tomato')
ax2.set_xlabel('Époque')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Précision — CIFAR-10 test')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Résultats finaux (epoch {N_C10}) :')
print(f'  MLP : {hist_c10["mlp_acc"][-1] * 100:.2f}%')
print(f'  CNN : {hist_c10["cnn_acc"][-1] * 100:.2f}%')
print(f'  Ecart : +{(hist_c10["cnn_acc"][-1] - hist_c10["mlp_acc"][-1]) * 100:.2f} pp pour le CNN')


### Observation

Sur CIFAR-10, l'écart entre MLP et CNN est **bien plus marqué** que sur MNIST :

| Dataset | MLP | CNN | Écart |
|---------|-----|-----|-------|
| MNIST (28×28 gris) | ~97.7% | ~99.0% | +1.3 pp |
| CIFAR-10 (32×32 RGB) | ~53% | ~74% | ~21 pp |

**Pourquoi ?** Les images naturelles ont une structure spatiale complexe :
- Textures, contours, objets à différentes positions
- Un MLP doit mémoriser *chaque* position séparément
- Le CNN extrait des **features hiérarchiques** réutilisables partout dans l'image